# Assignment 06 · Notebook 02: KKBox Churn (WSDM Cup 2018)

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

KKBox là dịch vụ nghe nhạc trực tuyến ở Đài Loan. Nhãn `is_churn` (tệp `train_v2.csv`) cho biết người dùng có
**không gia hạn** gói thuê bao trong 30 ngày sau khi hết hạn vào tháng 3/2017 hay không. Tệp `user_logs_v2.csv`
ghi hành vi nghe theo ngày của tháng 3/2017. Mỗi người dùng thành một chuỗi 31 ngày × 8 đặc trưng.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from scipy import stats
from a06.data_churn import FEATURES, cache_info, load_churn
from a06.export import save_metrics, write_dat, write_matrix_dat

info = cache_info()          # gom lại từ tệp CSV gốc nếu chưa có cache (khoảng 1 phút)
info

{'n_labeled': 970960,
 'n_with_log': 754551,
 'n_sample': 100000,
 'churn_rate_labeled': 0.08994191315811156,
 'churn_rate_with_log': 0.08964271467402468,
 'churn_rate_sample': 0.08964,
 'log_rows': 18396362,
 'log_rows_kept': 1795032,
 'secs_clipped': 379,
 'n_train': 70000,
 'n_val': 15000,
 'n_test': 15000}

## Lấy mẫu

Giữ người dùng vừa có nhãn vừa có ít nhất một dòng log, rồi lấy mẫu phân tầng 100.000 người (seed 42), giữ nguyên
tỉ lệ churn. Một ngày có thể có nhiều dòng log (nhiều thiết bị) nên cộng dồn theo (người, ngày); tổng số giây nghe
âm hoặc vượt 86.400 giây/ngày là lỗi ghi log và bị kẹp về [0, 86.400].

In [2]:
C = load_churn()
X, y = C["raw"], C["y_all"]
print("tensor:", X.shape, "· churn:", int(y.sum()), f"({y.mean():.2%})")
for k in ("train", "val", "test"):
    print(f"{k:5s} {len(C['idx'][k]):6d} người · churn {C['y'][k].mean():.4f}")

tensor: (100000, 31, 8) · churn: 8964 (8.96%)
train  70000 người · churn 0.0896
val    15000 người · churn 0.0897
test   15000 người · churn 0.0896


In [3]:
loyal, churn = X[y == 0], X[y == 1]
days = np.arange(1, 32)
fi = {f: FEATURES.index(f) for f in FEATURES}
write_dat("kkbox_traj", {"day": days,
                         "loyal_secs": loyal[:, :, fi["total_secs"]].mean(0), "churn_secs": churn[:, :, fi["total_secs"]].mean(0),
                         "loyal_unq": loyal[:, :, fi["num_unq"]].mean(0), "churn_unq": churn[:, :, fi["num_unq"]].mean(0),
                         "loyal_act": loyal[:, :, fi["active"]].mean(0), "churn_act": churn[:, :, fi["active"]].mean(0)})
active_days = X[:, :, fi["active"]].sum(1)
print("số ngày có nghe TB: loyal", active_days[y == 0].mean().round(2), "· churn", active_days[y == 1].mean().round(2))

số ngày có nghe TB: loyal 18.14 · churn 16.01


In [4]:
# Tổng số giờ nghe trong tháng của mỗi người (log1p), hai nhóm
hours = np.expm1(X[:, :, fi["total_secs"]]).sum(1) / 3600
lh = np.log1p(hours)
edges = np.linspace(0, lh.max(), 41)
mid = (edges[:-1] + edges[1:]) / 2
write_dat("kkbox_hist_hours", {"x": mid, "loyal": np.histogram(lh[y == 0], edges, density=True)[0],
                               "churn": np.histogram(lh[y == 1], edges, density=True)[0]})
hours_med = {"loyal": np.median(hours[y == 0]), "churn": np.median(hours[y == 1])}
print("trung vị giờ nghe/tháng:", {k: round(float(v), 2) for k, v in hours_med.items()})

trung vị giờ nghe/tháng: {'loyal': 23.32, 'churn': 19.95}


## Kiểm định t (Welch) từng đặc trưng

Mỗi người lấy trung bình 31 ngày của từng đặc trưng; so hai nhóm loyal và churn bằng t-test Welch (không giả định
phương sai bằng nhau).

In [5]:
per_user = X.mean(1)
ttest = {}
for f, j in fi.items():
    a, b = per_user[y == 0, j], per_user[y == 1, j]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    ttest[f] = {"loyal": a.mean(), "churn": b.mean(), "diff_pct": 100 * (b.mean() / a.mean() - 1), "t": t, "p": p}
pd.DataFrame(ttest).T

,loyal,churn,diff_pct,t,p
num_25,0.741870,0.687761,-7.293624,7.537065,5.192996e-14
num_50,0.332260,0.316060,-4.875588,4.417591,1.007852e-05
num_75,0.258375,0.243569,-5.730492,5.046233,4.579424e-07
num_985,0.275361,0.257222,-6.587214,5.587581,2.358068e-08
num_100,1.642573,1.455533,-11.387003,14.865788,0.000000e+00
num_unq,1.687524,1.513091,-10.336656,13.999397,0.000000e+00
total_secs,4.859126,4.296177,-11.585390,17.754343,0.000000e+00
active,0.585202,0.516492,-11.741114,19.367575,0.000000e+00


## Tương quan từng đặc trưng với nhãn churn

Tương quan điểm-nhị phân (Pearson với nhãn 0/1) theo ba cách tóm tắt chuỗi 31 ngày: trung bình cả tháng, trung bình
7 ngày cuối, và **xu hướng** = trung bình 7 ngày cuối trừ 7 ngày đầu. Nếu tín hiệu nằm ở sự thay đổi theo thời gian,
cột xu hướng sẽ mạnh hơn cột trung bình.

In [6]:
last7, first7 = X[:, -7:].mean(1), X[:, :7].mean(1)
views = {"mean": per_user, "last7": last7, "trend": last7 - first7}
label_corr = {f: {v: float(np.corrcoef(arr[:, j], y)[0, 1]) for v, arr in views.items()} for f, j in fi.items()}
write_dat("kkbox_label_corr", {"i": np.arange(len(FEATURES)), **{v: [label_corr[f][v] for f in FEATURES] for v in views}})
pd.DataFrame(label_corr).T.round(3)

,mean,last7,trend
num_25,-0.024,-0.063,-0.106
num_50,-0.014,-0.049,-0.082
num_75,-0.016,-0.048,-0.079
num_985,-0.017,-0.047,-0.076
num_100,-0.047,-0.095,-0.143
num_unq,-0.045,-0.097,-0.150
total_secs,-0.057,-0.115,-0.165
active,-0.063,-0.124,-0.171


In [7]:
corr = np.corrcoef(per_user.T)
write_matrix_dat("kkbox_corr", corr)
labels = {"total": int(info["n_sample"]), "churn": int(y.sum()), "loyal": int((y == 0).sum())}
save_metrics("kkbox", {**info, "labels": labels, "T": X.shape[1], "D": X.shape[2],
                       "active_days": {"loyal": active_days[y == 0].mean(), "churn": active_days[y == 1].mean()},
                       "hours_med": hours_med, "ttest": ttest, "label_corr": label_corr,
                       "corr_25_100": corr[fi["num_25"], fi["num_100"]], "corr_100_secs": corr[fi["num_100"], fi["total_secs"]]})
print("đã ghi outputs/metrics/kkbox.json")
pd.DataFrame(corr, index=FEATURES, columns=FEATURES).round(2)

đã ghi outputs/metrics/kkbox.json


,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs,active
num_25,1.00,0.89,0.87,0.82,0.70,0.79,0.76,0.75
num_50,0.89,1.00,0.94,0.87,0.66,0.74,0.70,0.68
num_75,0.87,0.94,1.00,0.89,0.67,0.74,0.70,0.68
num_985,0.82,0.87,0.89,1.00,0.66,0.72,0.69,0.66
num_100,0.70,0.66,0.67,0.66,1.00,0.97,0.97,0.91
num_unq,0.79,0.74,0.74,0.72,0.97,1.00,0.97,0.93
total_secs,0.76,0.70,0.70,0.69,0.97,0.97,1.00,0.99
active,0.75,0.68,0.68,0.66,0.91,0.93,0.99,1.00
